## Note: Native FAIR engine (epic #324)

This notebook was re-validated against the native, pyfair-free FAIR engine in
epic #324 (Task 10). The engine entry point is `NativeControlAwareRiskCalculator`
(consuming `FAIRParameters` / `FAIRDistribution`), replacing the legacy
`ControlAwareRiskCalculator` + dict-based `RiskParameters` API and the removed
`calculate_baseline_risk` / `monte_carlo_control_simulation` / `scenario_analysis`
helpers. Outputs below are produced by re-executing the notebook (papermill smoke
test `tests/smoke/test_notebooks.py`).

# FAIR CAM (Controls Analytics Model) - Comprehensive Example

This notebook demonstrates the FAIR CAM capabilities on the native, pyfair-free
Monte-Carlo engine (epic #324): quantitative control effectiveness modeling,
Boolean-group control composition, and closed-form per-control attribution.

## What's covered:

1. Defining cybersecurity controls (FAIR-CAM sub-function assignments)
2. Calculating control effectiveness
3. Defining a base FAIR risk scenario (`FAIRParameters`)
4. Control-enhanced risk (native `NativeControlAwareRiskCalculator`)
5. Monte Carlo distributions (from the engine's per-iteration ALE arrays)
6. Scenario analysis (per-scenario native engine runs)

In [ ]:
# Import required libraries
import sys
import os
import numpy as np
from datetime import datetime, timedelta

# matplotlib / pandas are illustrative-only and are NOT riskflow dependencies.
# Guard them so the engine-validation cells (the regression-critical surface the
# papermill smoke test protects) run even when plotting libs are absent.
try:
    import matplotlib.pyplot as plt
    import pandas as pd
    HAS_PLOTTING = True
except ModuleNotFoundError:
    plt = None
    pd = None
    HAS_PLOTTING = False
    print("matplotlib/pandas not installed - skipping plots (engine cells still run)")

# Import FAIR CAM modules (epic #324: native pyfair-free engine).
# The engine is now NativeControlAwareRiskCalculator, consuming FAIRParameters
# (FAIRDistribution-typed fields) instead of the legacy dict-based RiskParameters.
from fair_cam import (
    Control, ControlDomain, ControlRegistry, ControlType, ComplexityLevel,
    ControlEnhancedRisk, ControlAdjustment,
    ControlEffectivenessCalculator,
    NativeControlAwareRiskCalculator,
)
from fair_cam.risk_engine.fair_core import FAIRParameters, FAIRDistribution, DistributionType
from fair_cam.models.control import CostModel, EffectivenessMetric, FairCamControlFunctionAssignment
from fair_cam.models.sub_function import FairCamSubFunction

print('Libraries loaded successfully (epic #324 native engine)')


## 1. Define Cybersecurity Controls

First, we'll create a comprehensive set of cybersecurity controls based on industry frameworks like NIST 800-53 and CIS Controls.

In [ ]:
# Initialize control registry
control_registry = ControlRegistry()

# Enterprise firewall control
firewall_control = Control(
    control_id="FW-001",
    name="Enterprise Network Firewall",
    description="Next-generation firewall with deep packet inspection and threat intelligence",
    domain=ControlDomain.LOSS_EVENT,
    control_type=ControlType.TECHNICAL,
    assignments=[
        FairCamControlFunctionAssignment(
            sub_function=FairCamSubFunction.LEC_PREV_RESISTANCE,
            capability_value=0.85, coverage=0.90, reliability=0.95,
        ),
        FairCamControlFunctionAssignment(
            sub_function=FairCamSubFunction.LEC_DET_VISIBILITY,
            capability_value=0.75, coverage=0.80, reliability=0.90,
        ),
    ],
    nist_mappings=["SC-7", "SC-7(5)", "SI-4"],
    cis_mappings=["CIS-9.4", "CIS-12.1"],
    cost_model=CostModel(annual_cost=150_000),
    response_time_seconds=5,
    recovery_time_hours=2,
    tags=["network_security", "perimeter_defense", "critical"],
)

# Multi-factor authentication control
mfa_control = Control(
    control_id="IAM-001",
    name="Multi-Factor Authentication",
    description="Enterprise MFA solution with hardware tokens and biometrics",
    domain=ControlDomain.LOSS_EVENT,
    control_type=ControlType.TECHNICAL,
    assignments=[
        FairCamControlFunctionAssignment(
            sub_function=FairCamSubFunction.LEC_PREV_RESISTANCE,
            capability_value=0.92, coverage=0.95, reliability=0.98,
        ),
    ],
    nist_mappings=["IA-2", "IA-2(1)", "IA-2(2)"],
    cis_mappings=["CIS-6.3", "CIS-6.4"],
    cost_model=CostModel(annual_cost=90_000),
    response_time_seconds=30,
    recovery_time_hours=4,
    tags=["identity_management", "authentication", "critical"],
)

# Endpoint detection and response control
edr_control = Control(
    control_id="EDR-001",
    name="Endpoint Detection and Response",
    description="Advanced EDR solution with behavioral analysis and automated response",
    domain=ControlDomain.LOSS_EVENT,
    control_type=ControlType.TECHNICAL,
    assignments=[
        FairCamControlFunctionAssignment(
            sub_function=FairCamSubFunction.LEC_PREV_RESISTANCE,
            capability_value=0.80, coverage=0.85, reliability=0.90,
        ),
        FairCamControlFunctionAssignment(
            sub_function=FairCamSubFunction.LEC_DET_RECOGNITION,
            capability_value=0.85, coverage=0.90, reliability=0.92,
        ),
    ],
    nist_mappings=["SI-4", "SI-4(2)", "IR-4"],
    cis_mappings=["CIS-8.1", "CIS-8.2", "CIS-12.4"],
    cost_model=CostModel(annual_cost=120_000),
    response_time_seconds=15,
    recovery_time_hours=1,
    tags=["endpoint_security", "detection", "response"],
)

# Security awareness training (variance management control)
training_control = Control(
    control_id="VM-001",
    name="Security Awareness Training",
    description="Comprehensive security awareness and phishing simulation program",
    domain=ControlDomain.VARIANCE_MANAGEMENT,
    control_type=ControlType.ADMINISTRATIVE,
    assignments=[
        FairCamControlFunctionAssignment(
            sub_function=FairCamSubFunction.DSC_PREV_COMMUNICATION,
            capability_value=0.70, coverage=0.80, reliability=0.85,
        ),
    ],
    nist_mappings=["AT-2", "AT-3", "PM-13"],
    cis_mappings=["CIS-14.1", "CIS-14.2"],
    cost_model=CostModel(annual_cost=150_000),
    response_time_seconds=0,
    recovery_time_hours=0,
    tags=["human_factor", "awareness", "training"],
)

# Security dashboard (decision support control)
dashboard_control = Control(
    control_id="DS-001",
    name="Security Operations Dashboard",
    description="Real-time security metrics and threat intelligence dashboard",
    domain=ControlDomain.DECISION_SUPPORT,
    control_type=ControlType.TECHNICAL,
    assignments=[
        FairCamControlFunctionAssignment(
            sub_function=FairCamSubFunction.LEC_DET_VISIBILITY,
            capability_value=0.80, coverage=0.85, reliability=0.90,
        ),
    ],
    nist_mappings=["PM-5", "PM-6", "SI-4(5)"],
    cis_mappings=["CIS-12.5", "CIS-16.6"],
    cost_model=CostModel(annual_cost=55_000),
    response_time_seconds=1,
    recovery_time_hours=0.5,
    tags=["visibility", "decision_support", "metrics"],
)

# Register all controls
controls = [firewall_control, mfa_control, edr_control, training_control, dashboard_control]
for control in controls:
    control_registry.register_control(control)

print(f"Registered {len(controls)} cybersecurity controls:")
for control in controls:
    print(f"  - {control.name} ({control.domain.value})")


## 2. Calculate Control Effectiveness

Now we'll use the FAIR CAM effectiveness calculator to analyze each control's impact on risk factors.

In [ ]:
# Initialize effectiveness calculator
effectiveness_calc = ControlEffectivenessCalculator()

# Calculate base effectiveness for each control
effectiveness_results = {}
for control in controls:
    base_effectiveness = control.calculate_risk_reduction_factor()
    effectiveness_results[control.control_id] = {
        'name': control.name,
        'domain': control.domain.value,
        'base_effectiveness': base_effectiveness,
        'current_effectiveness': control.calculate_risk_reduction_factor(),
        'risk_reduction_factor': control.calculate_risk_reduction_factor(),
    }

print("Control Effectiveness Analysis:")
for cid, r in effectiveness_results.items():
    print(f"  {r['name'][:40]:40s} [{r['domain']:20s}] {r['base_effectiveness']:.3f}")

# Visualize control effectiveness (illustrative; requires matplotlib/pandas).
if HAS_PLOTTING:
    effectiveness_df = pd.DataFrame(effectiveness_results).T

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

    # Plot 1: Effectiveness by Domain
    domain_avg = effectiveness_df.groupby('domain')['base_effectiveness'].mean()
    ax1.bar(domain_avg.index, domain_avg.values, color=['#2E8B57', '#4169E1', '#DC143C'])
    ax1.set_title('Average Control Effectiveness by Domain')
    ax1.set_ylabel('Effectiveness Score')
    ax1.set_ylim(0, 1)
    ax1.tick_params(axis='x', rotation=45)

    # Plot 2: Individual Control Effectiveness
    control_names = [
        result['name'][:15] + '...' if len(result['name']) > 15 else result['name']
        for result in effectiveness_results.values()
    ]
    effectiveness_values = [r['base_effectiveness'] for r in effectiveness_results.values()]
    ax2.bar(range(len(control_names)), effectiveness_values,
            color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FECA57'])
    ax2.set_title('Individual Control Effectiveness')
    ax2.set_ylabel('Effectiveness Score')
    ax2.set_ylim(0, 1)
    ax2.set_xticks(range(len(control_names)))
    ax2.set_xticklabels(control_names, rotation=45, ha='right')

    plt.tight_layout()
    plt.show()


## 3. Define Base Risk Scenario

We'll define a realistic cyber risk scenario for a large enterprise organization.

In [ ]:
# Define base risk parameters for a data breach scenario.
# Native engine consumes FAIRParameters with FAIRDistribution-typed fields
# (epic #324). TEF / loss magnitudes use PERT; vulnerability uses a UNIFORM band.
base_risk_params = FAIRParameters(
    threat_event_frequency=FAIRDistribution(
        DistributionType.PERT,
        {'low': 2, 'mode': 6, 'high': 24},  # 2-24 serious attempts/year, mode 6
    ),
    # 65% central exploitation probability without controls, modeled as a band.
    vulnerability=FAIRDistribution(
        DistributionType.UNIFORM, {'low': 0.55, 'high': 0.75}
    ),
    primary_loss=FAIRDistribution(
        DistributionType.PERT,
        {'low': 2_000_000, 'mode': 8_000_000, 'high': 50_000_000},
    ),
    secondary_loss=FAIRDistribution(
        DistributionType.PERT,
        {'low': 1_000_000, 'mode': 4_000_000, 'high': 20_000_000},
    ),
)

N_SIMULATIONS = 10_000
RANDOM_SEED = 42

tef_p = base_risk_params.threat_event_frequency.parameters
pl_p = base_risk_params.primary_loss.parameters
sl_p = base_risk_params.secondary_loss.parameters
vuln_p = base_risk_params.vulnerability.parameters
print("Base Risk Scenario: Large Enterprise Data Breach")
print(f"  Threat Event Frequency: {tef_p['low']}-{tef_p['high']} events/year (mode {tef_p['mode']})")
print(f"  Base Vulnerability: {vuln_p['low']*100:.0f}%-{vuln_p['high']*100:.0f}% (uniform band)")
print(f"  Primary Loss Range: ${pl_p['low']:,} - ${pl_p['high']:,}")
print(f"  Secondary Loss Range: ${sl_p['low']:,} - ${sl_p['high']:,}")


## 4. Calculate Control-Enhanced Risk

Now we'll calculate risk with and without controls to demonstrate the FAIR CAM capability.

In [ ]:
# Initialize the native control-aware risk calculator (epic #324).
# A single calculator instance with a fixed seed gives reproducible streams;
# baseline = the same calculator run with NO active controls.
risk_calculator = NativeControlAwareRiskCalculator(
    control_registry, n_simulations=N_SIMULATIONS, random_seed=RANDOM_SEED
)

# Baseline risk (no controls): calculate_control_enhanced_risk([]) returns a
# ControlEnhancedRisk whose .base_risk == .residual_risk (no controls applied);
# we take .base_risk as the baseline FAIRRisk.
baseline_enhanced = risk_calculator.calculate_control_enhanced_risk(
    base_risk_params, [], "Baseline (No Controls)"
)
baseline_risk = baseline_enhanced.base_risk

print("Baseline Risk (No Controls):")
print(f"  Annualized Loss Expectancy: ${baseline_risk.annualized_loss_expectancy:,.0f}")
print(f"  95% Value at Risk: ${baseline_risk.var_95:,.0f}")
print(f"  99% Value at Risk: ${baseline_risk.var_99:,.0f}")
print(f"  Standard Deviation: ${baseline_risk.std_deviation:,.0f}")

# Risk with all controls active.
all_control_ids = [control.control_id for control in controls]
enhanced_risk_all = risk_calculator.calculate_control_enhanced_risk(
    base_risk_params,
    all_control_ids,
    "All Controls Active",
)

# Risk-reduction % is computed against the baseline ALE (the native
# ControlEnhancedRisk.calculate_total_risk_reduction is base-vs-residual within
# the same enhanced result; here we report against the separate baseline run).
total_reduction = (
    baseline_risk.annualized_loss_expectancy
    - enhanced_risk_all.residual_risk.annualized_loss_expectancy
)
print("\nRisk with All Controls:")
print(f"  Annualized Loss Expectancy: ${enhanced_risk_all.residual_risk.annualized_loss_expectancy:,.0f}")
print(f"  95% Value at Risk: ${enhanced_risk_all.residual_risk.var_95:,.0f}")
print(f"  Total Risk Reduction: ${total_reduction:,.0f}")
print(f"  Risk Reduction %: {(total_reduction / baseline_risk.annualized_loss_expectancy * 100):.1f}%")
print(f"  Total Control Cost: ${enhanced_risk_all.calculate_total_control_cost():,.0f}")
print(f"  Aggregate ROI: {enhanced_risk_all.calculate_aggregate_roi():.1f}%")

## 6. Monte Carlo Distributions (native engine)

The native `NativeControlAwareRiskCalculator` runs a Monte Carlo simulation
internally on every `calculate_control_enhanced_risk` call and exposes the full
per-iteration ALE arrays on `base_risk.simulation_results` /
`residual_risk.simulation_results`. We visualise those distributions directly
(epic #324 removed the legacy `monte_carlo_control_simulation` helper).

In [ ]:
# The native engine already produced full per-iteration ALE arrays. Re-use them
# (no separate monte_carlo_control_simulation call exists post epic #324).
baseline_ale_samples = np.asarray(baseline_risk.simulation_results, dtype=float)
residual_ale_samples = np.asarray(
    enhanced_risk_all.residual_risk.simulation_results, dtype=float
)
risk_reduction_samples = baseline_ale_samples - residual_ale_samples

baseline_ale_mean = float(np.mean(baseline_ale_samples))
residual_ale_mean = float(np.mean(residual_ale_samples))
risk_reduction_mean = float(np.mean(risk_reduction_samples))

print(f"Monte Carlo Results ({len(baseline_ale_samples):,} iterations):")
print(f"  Baseline ALE: ${baseline_ale_mean:,.0f} (+/-${np.std(baseline_ale_samples):,.0f})")
print(f"  Residual ALE: ${residual_ale_mean:,.0f} (+/-${np.std(residual_ale_samples):,.0f})")
print(f"  Risk Reduction: ${risk_reduction_mean:,.0f} (+/-${np.std(risk_reduction_samples):,.0f})")

baseline_95ci = np.percentile(baseline_ale_samples, [2.5, 97.5])
residual_95ci = np.percentile(residual_ale_samples, [2.5, 97.5])
reduction_95ci = np.percentile(risk_reduction_samples, [2.5, 97.5])

print(f"\n95% Confidence Intervals:")
print(f"  Baseline ALE: ${baseline_95ci[0]:,.0f} - ${baseline_95ci[1]:,.0f}")
print(f"  Residual ALE: ${residual_95ci[0]:,.0f} - ${residual_95ci[1]:,.0f}")
print(f"  Risk Reduction: ${reduction_95ci[0]:,.0f} - ${reduction_95ci[1]:,.0f}")

# Per-control standalone effectiveness (closed-form attribution, from the native
# ControlAdjustment list on the enhanced result).
adj_by_id = {a.control_id: a for a in enhanced_risk_all.control_adjustments}
control_names = []
control_reductions = []
for control in controls:
    adj = adj_by_id.get(control.control_id)
    control_names.append(control.name[:15] + '...' if len(control.name) > 15 else control.name)
    control_reductions.append(adj.risk_reduction_value if adj is not None else 0.0)

print("\nPer-control closed-form risk-reduction value:")
for name, red in zip(control_names, control_reductions):
    print(f"  {name:20s} ${red:,.0f}")

if HAS_PLOTTING:
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

    ax1.hist(baseline_ale_samples, bins=50, alpha=0.7, label='Baseline Risk', color='red')
    ax1.hist(residual_ale_samples, bins=50, alpha=0.7, label='Residual Risk', color='green')
    ax1.set_title('Annual Loss Expectancy Distribution')
    ax1.set_xlabel('ALE ($)')
    ax1.set_ylabel('Frequency')
    ax1.legend()
    ax1.ticklabel_format(style='plain', axis='x')

    ax2.hist(risk_reduction_samples, bins=50, alpha=0.7, color='blue')
    ax2.axvline(risk_reduction_mean, color='red', linestyle='--', label=f'Mean: ${risk_reduction_mean:,.0f}')
    ax2.axvline(reduction_95ci[0], color='orange', linestyle='--', alpha=0.7)
    ax2.axvline(reduction_95ci[1], color='orange', linestyle='--', alpha=0.7, label='95% CI')
    ax2.set_title('Risk Reduction Distribution')
    ax2.set_xlabel('Risk Reduction ($)')
    ax2.set_ylabel('Frequency')
    ax2.legend()
    ax2.ticklabel_format(style='plain', axis='x')

    x_pos = range(len(control_names))
    ax3.bar(x_pos, control_reductions,
            color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FECA57'])
    ax3.set_title('Per-Control Closed-Form Risk-Reduction Value')
    ax3.set_ylabel('Risk Reduction ($)')
    ax3.set_xticks(x_pos)
    ax3.set_xticklabels(control_names, rotation=45, ha='right')
    ax3.ticklabel_format(style='plain', axis='y')

    control_costs = [control.cost_model.annual_cost for control in controls]
    ax4.scatter(control_costs, control_reductions, s=100, alpha=0.7,
               c=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FECA57'])
    ax4.set_title('Annual Cost vs Risk-Reduction Value')
    ax4.set_xlabel('Annual Cost ($)')
    ax4.set_ylabel('Risk Reduction ($)')
    ax4.ticklabel_format(style='plain', axis='both')
    for i, control in enumerate(controls):
        ax4.annotate(control.name[:10], (control_costs[i], control_reductions[i]),
                    xytext=(5, 5), textcoords='offset points', fontsize=8)

    plt.tight_layout()
    plt.show()


## 8. Scenario Analysis (native per-scenario runs)

Analyze different control configurations by running each through the native
engine directly (epic #324 removed the legacy `scenario_analysis` helper).

In [ ]:
# Define multiple scenarios for comparison
scenarios = {
    "Minimum Viable": ["FW-001", "IAM-001"],
    "Balanced Approach": ["FW-001", "IAM-001", "VM-001"],
    "Technical Focus": ["FW-001", "IAM-001", "EDR-001", "DS-001"],
    "Comprehensive": ["FW-001", "IAM-001", "EDR-001", "VM-001", "DS-001"],
}

print("Analyzing multiple risk scenarios (native per-scenario engine runs)...")

# Epic #324 removed the legacy scenario_analysis() helper; run each scenario
# through calculate_control_enhanced_risk directly. Each call spawns its own
# independent seed stream from the calculator's SeedSequence.
scenario_results = {
    name: risk_calculator.calculate_control_enhanced_risk(
        base_risk_params, control_ids, name
    )
    for name, control_ids in scenarios.items()
}

# Build a plain-dict comparison table (no pandas dependency).
scenario_comparison = []
for scenario_name, enhanced_risk in scenario_results.items():
    residual_ale = enhanced_risk.residual_risk.annualized_loss_expectancy
    reduction = baseline_risk.annualized_loss_expectancy - residual_ale
    scenario_comparison.append({
        'Scenario': scenario_name,
        'Controls Count': len(scenarios[scenario_name]),
        'Annual Cost': enhanced_risk.calculate_total_control_cost(),
        'Residual ALE': residual_ale,
        'Risk Reduction': reduction,
        'ROI (%)': enhanced_risk.calculate_aggregate_roi(),
        'Risk Reduction (%)': (reduction / baseline_risk.annualized_loss_expectancy) * 100,
    })

print("\nScenario Comparison:")
print(f"  {'Scenario':20s} {'#':>2s} {'AnnualCost':>14s} {'ResidualALE':>16s} {'Reduction':>16s} {'ROI%':>8s} {'Red%':>6s}")
for row in scenario_comparison:
    print(f"  {row['Scenario']:20s} {row['Controls Count']:>2d} "
          f"${row['Annual Cost']:>13,.0f} ${row['Residual ALE']:>15,.0f} "
          f"${row['Risk Reduction']:>15,.0f} {row['ROI (%)']:>7.1f} {row['Risk Reduction (%)']:>5.1f}")

if HAS_PLOTTING:
    scenario_df = pd.DataFrame(scenario_comparison).round(1)
    scenarios_list = list(scenarios.keys())
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']

    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

    costs = scenario_df['Annual Cost'].values
    risk_reductions = scenario_df['Risk Reduction'].values
    ax1.scatter(costs, risk_reductions, s=200, alpha=0.7, c=colors)
    ax1.set_title('Cost vs Risk Reduction by Scenario')
    ax1.set_xlabel('Annual Cost ($)')
    ax1.set_ylabel('Risk Reduction ($)')
    ax1.ticklabel_format(style='plain', axis='both')
    for i, scenario in enumerate(scenarios_list):
        ax1.annotate(scenario, (costs[i], risk_reductions[i]),
                    xytext=(5, 5), textcoords='offset points', fontsize=10)

    ax2.bar(range(len(scenarios_list)), scenario_df['ROI (%)'].values, color=colors)
    ax2.set_title('Return on Investment by Scenario')
    ax2.set_ylabel('ROI (%)')
    ax2.set_xticks(range(len(scenarios_list)))
    ax2.set_xticklabels(scenarios_list, rotation=45, ha='right')
    ax2.axhline(y=0, color='black', linestyle='--', alpha=0.5)

    ax3.bar(range(len(scenarios_list)), scenario_df['Residual ALE'].values, color=colors)
    ax3.set_title('Residual Annual Loss Expectancy')
    ax3.set_ylabel('Residual ALE ($)')
    ax3.set_xticks(range(len(scenarios_list)))
    ax3.set_xticklabels(scenarios_list, rotation=45, ha='right')
    ax3.ticklabel_format(style='plain', axis='y')

    ax4.bar(range(len(scenarios_list)), scenario_df['Risk Reduction (%)'].values, color=colors)
    ax4.set_title('Risk Reduction Percentage')
    ax4.set_ylabel('Risk Reduction (%)')
    ax4.set_xticks(range(len(scenarios_list)))
    ax4.set_xticklabels(scenarios_list, rotation=45, ha='right')
    ax4.set_ylim(0, 100)

    plt.tight_layout()
    plt.show()

# Recommend the scenario with the highest ROI (plain-list argmax; no pandas).
roi_values = [row['ROI (%)'] for row in scenario_comparison]
best_scenario_idx = int(np.argmax(roi_values))
best_row = scenario_comparison[best_scenario_idx]
best_scenario = best_row['Scenario']

print(f"\nRecommended Scenario: {best_scenario}")
print(f"  Rationale: Highest ROI of {best_row['ROI (%)']:.1f}%")
print(f"  Controls: {', '.join(scenarios[best_scenario])}")
print(f"  Investment: ${best_row['Annual Cost']:,.0f}")
print(f"  Risk Reduction: {best_row['Risk Reduction (%)']:.1f}%")


## 10. Summary and Key Benefits

Let's summarize the key capabilities and benefits of FAIR CAM.

In [ ]:
# Generate summary report (epic #324 native engine)
print('='*80)
print('FAIR CAM (Controls Analytics Model) - Executive Summary')
print('='*80)

print('\nBUSINESS IMPACT:')
baseline_ale = baseline_risk.annualized_loss_expectancy
residual_ale = enhanced_risk_all.residual_risk.annualized_loss_expectancy
risk_reduction = baseline_ale - residual_ale
print(f'  Baseline Risk (No Controls): ${baseline_ale:,.0f}')
print(f'  Residual Risk (With Controls): ${residual_ale:,.0f}')
print(f'  Risk Reduction Achieved: ${risk_reduction:,.0f}')
print(f'  Risk Reduction Percentage: {(risk_reduction/baseline_ale)*100:.1f}%' if baseline_ale > 0 else '  N/A')

print('\nTECHNICAL CAPABILITIES:')
print('  - Native (pyfair-free) Monte-Carlo FAIR engine (epic #324)')
print('  - Quantitative Control Effectiveness Modeling (PR kappa multi-domain composition)')
print('  - Boolean-group control composition + closed-form per-control attribution')
print('  - IRIS-2025-grounded Industry Calibration')

print('\nMOST EFFECTIVE CONTROL (standalone closed-form risk-reduction value):')
best_ctrl = max(effectiveness_results.items(), key=lambda x: x[1]['base_effectiveness'])
print(f'  {best_ctrl[1]["name"]}: {best_ctrl[1]["base_effectiveness"]:.2f}')
